In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
drivers_df=spark.read\
    .option("inferSchema", True)\
        .json("abfss://demofiles@formula1adls.dfs.core.windows.net/source_files/drivers.json")
drivers_df.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType,DateType
name_schema= StructType(fields=
                        [
                            StructField("forename", StringType(), True),
                            StructField("surname",  StringType(), True)
                        ])
 = StructType(fields=
                             [
                             StructField("driverId", IntegerType(), False),
                             StructField("driverRef", StringType(), True),
                             StructField("number", IntegerType(), True),
                             StructField("code", StringType(), True),
                             StructField("dob", DateType(), True),
                             StructField("name", name_schema),
                             StructField("nationality", StringType(), True),
                             StructField("url", StringType(), True)
                             ])


In [0]:
drivers_df_1=spark.read\
    .schema(drivers_schema)\
        .json("abfss://demofiles@formula1adls.dfs.core.windows.net/source_files/drivers.json")


In [0]:
drivers_df_1.display()

In [0]:
drivers_df_1.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp,current_date
drivers_df_final= drivers_df_1.withColumn("ingestion_timestamp", current_timestamp())\
    .withColumn("ingestion_date", current_date())\
        .withColumnRenamed("driverId", "driver_id")\
            .withColumnRenamed("driverRef", "driver_ref")\
                .drop("url")


In [0]:
drivers_df_final.display()

In [0]:
drivers_df_final.write.mode("overwrite").format("delta").option("path", "abfss://raw@formula1adls.dfs.core.windows.net/drivers").saveAsTable(f"formula1_{env}.bronze.drivers")